What this code does:
Sets up a controlled Stable Diffusion (Runway's Stable Diffusion v1.5 model) generation with the bounded attention method.

In [ ]:
!pip install --upgrade pip
!pip install --pre xformers -f https://download.pytorch.org/whl/cu118/torch_stable.html
!pip install pytorch-lightning
!pip install torch torchvision torchaudio --upgrade
!pip install diffusers transformers accelerate safetensors pytorch-lightning
!pip install torch_kmeans
import torch
!pip install torch torchvision torchaudio
!pip install diffusers transformers accelerate safetensors
!pip install numpy matplotlib
import sys
import importlib.util
!pip install nltk
import nltk

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
!git clone https://github.com/omer11a/bounded-attention.git
%cd bounded-attention

In [ ]:
!pip install -r requirements.txt

In [ ]:
sys.path.append("/content/bounded-attention")
module_path = "/content/bounded-attention/run_sd.py"
spec = importlib.util.spec_from_file_location("run_sd", module_path)
run_sd = importlib.util.module_from_spec(spec)
sys.modules["run_sd"] = run_sd
spec.loader.exec_module(run_sd)
run = run_sd.run

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from diffusers import StableDiffusionPipeline

In [ ]:
# stable diffusion with BA boxes

model_name = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")
pipe.save_pretrained("sd_model")

In [ ]:
# Shark and dolphin prompt
from transformers import CLIPTokenizer

prompt = "A shark chasing a dolphin in the ocean"
prompt = "Please show me an image of a shark chasing a dolphin in the ocean"
prompt = "Shark chase dolphin, ocean."
prompt = "A shark with its mouth open, swimming behind and chasing a smiling dolphin in the blue ocean."
prompt = "In the ocean, a shark chasing a dolphin."
prompt = "A shark chasing a dolphin in the ocean, can you show me an image of that?"
prompt = "A dolphin being chased by a shark in the ocean"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[2,3,4,5,6], [13,14]] #shark, dolphin

boxes = [
    [0.1, 0.2, 0.45, 0.8],  # shark
    [0.55, 0.2, 0.9, 0.8]   # dolphin
]

In [ ]:
# flowers prompt
from transformers import CLIPTokenizer

prompt = "A porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers"
prompt = "Please show me an image of a porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers"
prompt = "Porcelain pot tulips, metal can orchids, glass jar sunflowers."
prompt = "A white porcelain pot with pink tulips and a silver metal can with purple orchids and a clear glass jar with yellow sunflowers, next to each other on a wooden table"
prompt = "In the kitchen, a porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers"
prompt = "A porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers, can you show me an image of that?"
prompt = "Tulips in a porcelain pot and orchids in a metal can and sunflowers in a glass jar"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[4,5], [1], [10,11], [7], [16,17], [13]]

boxes = [
    [0.1, 0.5, 0.25, 0.85],     # pot1
    [0.05, 0.05, 0.3, 0.5],     # flower1
    [0.45, 0.45, 0.6, 0.8],     # pot2
    [0.37, 0.15, 0.65, 0.45],   # flower2
    [0.75, 0.65, 0.9, 0.95],    # pot3
    [0.7, 0.1, 0.95, 0.65] ]     # flower3

In [ ]:
# cartoon people prompt
from transformers import CLIPTokenizer

prompt = "3D Pixar animation of a princess, an elf, and a fairy in a castle"
prompt = "Please show me a 3D Pixar animation of a princess, an elf, and a fairy in a castle"
prompt = "3D Pixar animation, princess, elf, fairy, castle"
prompt = "A 3D Pixar animation of a blonde princess wearing a pink dress next to a fat elf with pointed ears with a tan fairy wearing a green dress in a diamond castle"
prompt = "In a castle, a princess, an elf, and a fairy 3D Pixar animation"
prompt = "A princess, an elf, and a fairy in a castle, can you show me a 3D Pixar animation of that?"
prompt = "3D Pixar animation of a fairy, an elf, and a princess in a castle"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[7],[10],[14]]

boxes = [
    [0.1, 0.1, 0.3, 0.85],     # princess
    [0.4, 0.45, 0.7, 0.85],     # elf
    [0.7, 0.1, 0.85, 0.35]      # fairy
]

In [ ]:
# realistic person, solo prompt
from transformers import CLIPTokenizer

prompt = "A hyperrealistic jedi with a lightsaber in the death star."
prompt = "Please show me a hyperrealistic jedi with a lightsaber in the death star."
prompt = "Hyperrealistic jedi, lightsaber, death star."
prompt = "A hyperrealistic jedi with a yellow lightsaber and long hair at emperor palpatine’s throne in the death star."
prompt = "In the death star, a hyperrealistic jedi with a lightsaber"
prompt = "A hyperrealistic jedi with a lightsaber, can you show me an image of that?"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

subject_token_indices = [[2,3,4,5,6,7,8]]

boxes = [[0.4, 0.3, 0.6, 0.85]]

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# Bounded Attention with SD
import sys
import importlib.util

sys.path.append("/content/bounded-attention")

module_path = "/content/bounded-attention/run_sd.py"
spec = importlib.util.spec_from_file_location("run_sd", module_path)
run_sd = importlib.util.module_from_spec(spec)
sys.modules["run_sd"] = run_sd
spec.loader.exec_module(run_sd)

run = run_sd.run

run(
    boxes=boxes,
    prompt=prompt,
    subject_token_indices=subject_token_indices,
    out_dir="outputs",
    seed=42,
    batch_size=1,
    init_step_size=25,
    final_step_size=10
)